# Paper 2: Regime Labels Are Not Resolution-Invariant

This notebook demonstrates the **resolution-invariance** analysis from Paper 2 of the MRV framework.

**Core question:** Do regime labels (crisis vs. calm) agree across different data frequencies
(5-minute, 15-minute, 1-hour, daily)?

We test this using cross-frequency Adjusted Rand Index (ARI), permutation tests,
expanding-window regime fitting, time-of-day seasonality analysis, and more.

Based on *Zheng, Low & Wang (2026)*, submitted to **Finance Research Letters**.

## 1. Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mrv.data.reader import resample_ohlc
from mrv.validator.res import (
    analyze_asset, run_robustness, ResValidator,
    _features, _fit_regime_gmm, _align_regimes_to_5m, _compute_ari_matrix,
    _mean_offdiag, _permute_pvalue_mean_offdiag_ari,
    _block_permute_pvalue_mean_offdiag_ari,
    _gmm_fit_diagnostics, _fit_regime_expanding,
    _compute_tod_crisis_distribution,
    FREQS, DEFAULT_GMM_K, EPISODES
)
from mrv.validator.metrics import ARI_THRESHOLD

print(f"Frequencies under test: {FREQS}")
print(f"Default GMM components: {DEFAULT_GMM_K}")
print(f"ARI threshold for agreement: {ARI_THRESHOLD}")

## 2. Synthetic 5-Minute OHLCV Data

We generate 15 trading days of synthetic 5-minute OHLCV data (78 bars per day = 1170 bars).
Returns are drawn from a regime-switching process: 70% calm (low vol) and 30% crisis (high vol).

In [ ]:
np.random.seed(42)
n_days = 15
bars_per_day = 78  # SPY: 9:30-16:00, 5min bars

# Build business-day 5min index (America/New_York)
dates = pd.bdate_range("2026-01-05", periods=n_days)
idx_parts = []
for d in dates:
    start = pd.Timestamp(f"{d.date()} 09:30", tz="America/New_York")
    times = pd.date_range(start, periods=bars_per_day, freq="5min")
    idx_parts.append(times)
idx = idx_parts[0]
for part in idx_parts[1:]:
    idx = idx.append(part)

# Regime-switching returns
regime = np.random.choice([0, 1], size=len(idx), p=[0.7, 0.3])
returns = np.where(regime == 0,
                   np.random.randn(len(idx)) * 0.001,
                   np.random.randn(len(idx)) * 0.004)
close = 100.0 * np.exp(np.cumsum(returns))
high = close * (1 + np.abs(np.random.randn(len(idx))) * 0.001)
low = close * (1 - np.abs(np.random.randn(len(idx))) * 0.001)
open_ = close * (1 + np.random.randn(len(idx)) * 0.0005)

df_5m = pd.DataFrame({"Open": open_, "High": high, "Low": low, "Close": close}, index=idx)
print(f"Synthetic 5m OHLCV: {len(df_5m)} bars, {n_days} trading days")
print(f"Date range: {df_5m.index[0]} to {df_5m.index[-1]}")
df_5m.head(10)

## 3. Multi-Frequency Resampling

Resample the 5-minute data to 15-minute, 1-hour, and daily frequencies
using `mrv.data.reader.resample_ohlc()`. This preserves the correct OHLC semantics
(open=first, high=max, low=min, close=last).

In [ ]:
freq_map = {"5m": df_5m}
for freq in ["15m", "1h", "1d"]:
    freq_map[freq] = resample_ohlc(df_5m, freq)

print("Bar counts by frequency:")
for freq, df in freq_map.items():
    print(f"  {freq:>4s}: {len(df):>6d} bars")

print("\n15m sample:")
freq_map["15m"].head()

## 4. Feature Engineering & Regime Fitting

For each frequency we compute volatility features via `_features()` and then
fit a 2-component Gaussian Mixture Model via `_fit_regime_gmm()`.
The higher-volatility component is labelled *crisis* (1), the other *calm* (0).

In [ ]:
feat_map = {}
regime_map = {}

for freq, df in freq_map.items():
    feat = _features(df)
    feat_map[freq] = feat
    labels = _fit_regime_gmm(feat, k=DEFAULT_GMM_K)
    regime_map[freq] = labels
    crisis_share = labels.mean() if labels is not None else float("nan")
    print(f"{freq:>4s}: {len(feat)} obs, crisis share = {crisis_share:.2%}")

## 5. Align to 5-Minute Timestamps & Cross-Frequency ARI

Lower-frequency labels are broadcast back to 5-minute timestamps using
`_align_regimes_to_5m()`. We then compute the 4x4 ARI matrix with
`_compute_ari_matrix()`. ARI = 1 means perfect agreement; ARI near 0 means
labels are essentially random relative to each other.

In [ ]:
aligned = _align_regimes_to_5m(regime_map, df_5m.index)
ari_matrix = _compute_ari_matrix(aligned, FREQS)

print("Cross-frequency ARI matrix:")
print(pd.DataFrame(ari_matrix, index=FREQS, columns=FREQS).round(3))

mean_offdiag = _mean_offdiag(ari_matrix)
print(f"\nMean off-diagonal ARI: {mean_offdiag:.4f}")
print(f"ARI threshold: {ARI_THRESHOLD}")
print(f"Resolution-invariant? {'YES' if mean_offdiag >= ARI_THRESHOLD else 'NO'}")

## 6. ARI Heatmap

Visual representation of the cross-frequency ARI matrix.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(ari_matrix, vmin=-0.1, vmax=1.0, cmap="RdYlGn")
ax.set_xticks(range(len(FREQS)))
ax.set_yticks(range(len(FREQS)))
ax.set_xticklabels(FREQS)
ax.set_yticklabels(FREQS)
for i in range(len(FREQS)):
    for j in range(len(FREQS)):
        ax.text(j, i, f"{ari_matrix[i, j]:.2f}",
                ha="center", va="center", fontsize=12,
                color="white" if ari_matrix[i, j] < 0.4 else "black")
plt.colorbar(im, ax=ax, label="Adjusted Rand Index")
ax.set_title("Cross-Frequency ARI Matrix")
plt.tight_layout()
plt.show()

## 7. Permutation Tests

We run two permutation tests to assess statistical significance of the mean off-diagonal ARI:

1. **Standard permutation test** (`_permute_pvalue_mean_offdiag_ari`): shuffles labels independently.
2. **Block permutation test** (`_block_permute_pvalue_mean_offdiag_ari`): shuffles in temporal blocks
   to preserve autocorrelation structure.

A small p-value means the observed ARI is *higher* than chance -- the labels do carry
some cross-frequency information, even if ARI is below the agreement threshold.

In [ ]:
# Standard permutation test
p_std = _permute_pvalue_mean_offdiag_ari(aligned, FREQS, n_perm=500, seed=42)
print(f"Standard permutation p-value: {p_std:.4f}")

# Block permutation test (preserves temporal autocorrelation)
p_block = _block_permute_pvalue_mean_offdiag_ari(aligned, FREQS, n_perm=500, seed=42)
print(f"Block permutation p-value:    {p_block:.4f}")

print(f"\nInterpretation:")
if p_std < 0.05:
    print("  Standard test: ARI is significantly above chance (p < 0.05)")
else:
    print("  Standard test: ARI is NOT significantly above chance")
if p_block < 0.05:
    print("  Block test: ARI is significantly above chance even accounting for autocorrelation")
else:
    print("  Block test: ARI is NOT significantly above chance after accounting for autocorrelation")

## 8. GMM Fit Diagnostics

For each frequency we inspect BIC, AIC, mean separation (in standard deviations),
and component overlap. These diagnostics tell us whether the GMM is finding
genuinely distinct regimes.

In [ ]:
diag_rows = []
for freq in FREQS:
    feat = feat_map[freq]
    diag = _gmm_fit_diagnostics(feat, k=DEFAULT_GMM_K)
    diag["freq"] = freq
    diag_rows.append(diag)

diag_df = pd.DataFrame(diag_rows).set_index("freq")
print("GMM Fit Diagnostics by Frequency:")
print(diag_df.round(4))

## 9. Expanding-Window Regime Fitting (No Look-Ahead)

`_fit_regime_expanding()` re-fits the GMM at each time step using only data
available up to that point (no look-ahead bias). We compare the expanding-window
labels with the full-sample labels to see how stable regime assignments are.

In [ ]:
from sklearn.metrics import adjusted_rand_score

# Run expanding-window for the 5m frequency
expanding_labels = _fit_regime_expanding(feat_map["5m"], k=DEFAULT_GMM_K)

if expanding_labels is not None and regime_map["5m"] is not None:
    # Align lengths (expanding may drop early observations)
    n = min(len(expanding_labels), len(regime_map["5m"]))
    tail_exp = expanding_labels[-n:]
    tail_full = regime_map["5m"][-n:]
    mask = ~(np.isnan(tail_exp) | np.isnan(tail_full))
    ari_exp = adjusted_rand_score(tail_full[mask], tail_exp[mask])
    print(f"Expanding vs full-sample ARI (5m): {ari_exp:.4f}")
    print(f"Non-NaN observations: {mask.sum()} / {n}")
else:
    print("Expanding-window fitting returned None (insufficient data).")

## 10. Time-of-Day Seasonality

Intraday data often exhibits time-of-day patterns in volatility (e.g., the U-shaped
pattern). `_compute_tod_crisis_distribution()` computes the share of crisis labels
by hour of the trading day.

In [ ]:
tod = _compute_tod_crisis_distribution(regime_map["5m"], df_5m.index)

if tod is not None:
    print("Crisis share by hour:")
    print(tod.round(4))

    fig, ax = plt.subplots(figsize=(8, 4))
    tod.plot(kind="bar", ax=ax, color="indianred", edgecolor="black")
    ax.set_ylabel("Crisis Share")
    ax.set_xlabel("Hour of Day (ET)")
    ax.set_title("Time-of-Day Crisis Distribution (5m regime labels)")
    ax.axhline(regime_map["5m"].mean(), ls="--", color="grey", label="Overall crisis share")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("TOD distribution not available.")

## 11. Event vs. Calm Window Analysis

We split the sample into *event* windows (days with high realized volatility) and
*calm* windows, then compute separate ARI matrices for each sub-period.
If regime labels are resolution-invariant only during crises but not during calm
periods (or vice versa), this analysis will reveal it.

In [ ]:
# Identify event days: top 30% by daily realized volatility
daily_ret = df_5m["Close"].resample("1D").last().pct_change().dropna()
daily_vol = daily_ret.abs()
vol_threshold = daily_vol.quantile(0.7)
event_days = daily_vol[daily_vol >= vol_threshold].index
calm_days = daily_vol[daily_vol < vol_threshold].index

print(f"Event days (top 30% vol): {len(event_days)}")
print(f"Calm days:                {len(calm_days)}")

# Build masks for 5m index
event_mask = df_5m.index.normalize().isin(event_days.normalize())
calm_mask = df_5m.index.normalize().isin(calm_days.normalize())

for label, mask in [("EVENT", event_mask), ("CALM", calm_mask)]:
    sub_aligned = {f: a[mask[:len(a)]] for f, a in aligned.items()}
    sub_ari = _compute_ari_matrix(sub_aligned, FREQS)
    print(f"\n{label} window ARI matrix:")
    print(pd.DataFrame(sub_ari, index=FREQS, columns=FREQS).round(3))
    print(f"  Mean off-diagonal ARI: {_mean_offdiag(sub_ari):.4f}")

## 12. Using `analyze_asset` (All-in-One)

`analyze_asset()` is the high-level entry point that runs the full Paper 2 pipeline
for a single asset: resampling, feature extraction, regime fitting, ARI computation,
permutation tests, diagnostics, expanding-window fitting, and TOD analysis.

In [ ]:
result = analyze_asset("SyntheticAsset", df_5m)

print("analyze_asset() returned keys:")
for k, v in result.items():
    if isinstance(v, np.ndarray):
        print(f"  {k}: ndarray {v.shape}")
    elif isinstance(v, dict):
        print(f"  {k}: dict with {len(v)} entries")
    elif isinstance(v, pd.Series):
        print(f"  {k}: Series len={len(v)}")
    else:
        print(f"  {k}: {v}")

## 13. Robustness Sweep

`run_robustness()` repeats the analysis across different GMM component counts (K)
and feature window scales. This checks whether the resolution-invariance finding
is sensitive to modelling choices.

In [ ]:
rob = run_robustness(
    "SyntheticAsset",
    df_5m,
    ks=[2, 3],
    window_scales=[0.5, 1.0, 2.0],
)

rob_df = pd.DataFrame(rob)
print("Robustness sweep results:")
print(rob_df.to_string(index=False))

## 14. Using `ResValidator` (Config-Driven)

`ResValidator` wraps the full pipeline in a config-driven interface that can
be used programmatically or from the CLI.

In [ ]:
result = ResValidator({
    "validator": {
        "report_dir": "reports",
        "res": {
            "model": "gmm",
            "n_states": 2,
        },
    },
}).validate(prices={"SyntheticAsset": df_5m})

print("ResValidator result keys:", list(result.keys()))
for k, v in result.items():
    if isinstance(v, dict):
        print(f"  {k}: {list(v.keys())}")
    else:
        print(f"  {k}: {v}")

## 15. Config-File Workflow

For production use, define a YAML config and run via the CLI:

```yaml
# config.yaml
data:
  source: ib          # or csv
  symbols: [SPY]
  start: "2024-01-01"
  end: "2024-12-31"

validator:
  report_dir: reports
  res:
    model: gmm
    n_states: 2
    permutations: 1000
    robustness:
      ks: [2, 3, 4]
      window_scales: [0.5, 1.0, 2.0]
```

Then run from the command line:

```bash
python run.py run config.yaml res
```

This will:
1. Fetch or load OHLCV data for each symbol
2. Run the full resolution-invariance analysis
3. Generate reports (ARI heatmaps, diagnostics tables, TOD plots) in `reports/`

---

**Summary:** This notebook walked through the complete Paper 2 pipeline:

- Multi-frequency resampling and regime fitting
- Cross-frequency ARI matrix and heatmap
- Standard and block permutation tests for statistical significance
- GMM fit diagnostics (BIC, AIC, separation, overlap)
- Expanding-window regime fitting to avoid look-ahead bias
- Time-of-day crisis seasonality
- Event vs. calm sub-period analysis
- All-in-one `analyze_asset()` and robustness sweeps
- Config-driven `ResValidator` for production workflows